# Data preparation pipeline per U-Net (aloni di materia oscura)

Questo notebook costruisce una pipeline completa di preparazione dati a partire da:
- `snapshot_090.hdf5` (particelle DM)
- `groups_090.hdf5` (catalogo aloni)

Obiettivo: generare patch `256x256` con input SPH smoothed e target di segmentazione per addestrare una U-Net.

## 1) Setup e dipendenze

Importiamo le librerie e definiamo una funzione helper per creare cartelle di output.

In [ ]:
from pathlib import Path
import json
import math

import h5py
import matplotlib.pyplot as plt
import numpy as np

# Libreria indicata dal prof Benitez-Llambay
import sphviewer2


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

## 2) Configurazione

In questa sezione imposti i percorsi locali e i parametri fisici/numerici della pipeline.

Valori chiave dal meeting:
- `MASS_CUT = 2.6e9 Msun/h` (100 particelle)
- risoluzione SPH `1024x1024` (`r_max=10`)
- `k_neighbors=32`
- patch `256x256`

In [ ]:
# === Percorsi (modifica in base al tuo filesystem) ===
SNAPSHOT_PATH = Path(r'C:\\Users\\kekko\\Downloads\\Python\\TESI\\CAMELS_data\\snapshot_090.hdf5')
GROUPS_PATH = Path(r'C:\\Users\\kekko\\Downloads\\Python\\TESI\\CAMELS_data\\groups_090.hdf5')

# Salvataggio nella cartella corrente (nessuna creazione directory)
CACHE_ROOT = Path('.')
MAPS_ROOT = Path('.')
PATCH_ROOT = Path('.')

# === Parametri fisici ===
MASS_CUT = 2.6e9  # Msun/h
N_MIN_PARTICLES = 100

# === Parametri rendering SPH ===
SPH_RESOLUTION = 1024
SPH_R_MAX = 10
K_NEIGHBORS = 32
TARGET_CELLS_PER_H = 4

# === Parametri slicing ===
N_SLICES_Z = 25

# === Parametri patch ===
PATCH_SIZE = 256
PATCH_STRIDE = 256  # non-overlap

assert SPH_RESOLUTION % PATCH_SIZE == 0, 'PATCH_SIZE deve dividere SPH_RESOLUTION.'

config = {
    'snapshot_path': str(SNAPSHOT_PATH),
    'groups_path': str(GROUPS_PATH),
    'mass_cut_msun_h': MASS_CUT,
    'sph_resolution': SPH_RESOLUTION,
    'sph_r_max': SPH_R_MAX,
    'k_neighbors': K_NEIGHBORS,
    'target_cells_per_h': TARGET_CELLS_PER_H,
    'n_slices_z': N_SLICES_Z,
    'patch_size': PATCH_SIZE,
    'patch_stride': PATCH_STRIDE,
}
print(json.dumps(config, indent=2))

## 3) Lettura snapshot e catalogo aloni

Carichiamo:
- coordinate/velocità delle particelle DM (`PartType1`)
- massa particella da `MassTable[1]` (oppure dataset `Masses` se presente)
- catalogo aloni dal file gruppi, con filtri di massa.

In [ ]:
def load_dm_snapshot(snapshot_path: Path):
    with h5py.File(snapshot_path, 'r') as f:
        header = f['Header'].attrs
        box_size = float(header['BoxSize'])  # ckpc/h in TNG/CAMELS

        coords = f['PartType1']['Coordinates'][...].astype(np.float32)
        vels = f['PartType1']['Velocities'][...].astype(np.float32)

        if 'Masses' in f['PartType1']:
            masses = f['PartType1']['Masses'][...].astype(np.float32)
            m_particle = float(np.median(masses))
        else:
            m_particle = float(header['MassTable'][1])
            masses = np.full(coords.shape[0], m_particle, dtype=np.float32)

    return {
        'coords': coords,
        'vels': vels,
        'masses': masses,
        'm_particle': m_particle,
        'box_size': box_size,
    }


def pick_first_available(h5file, names):
    for name in names:
        if name in h5file:
            return h5file[name][...]
    raise KeyError(f'Nessuno dei dataset {names} trovato nel file gruppi.')


def load_halos(groups_path: Path, mass_cut: float):
    with h5py.File(groups_path, 'r') as f:
        # nomi possibili in base a variante del catalogo
        group_pos = pick_first_available(f, ['Group/GroupPos', 'GroupPos']).astype(np.float32)
        group_mass = pick_first_available(
            f,
            ['Group/Group_M_Crit200', 'Group_M_TopHat200', 'Group/GroupMass', 'GroupMass']
        ).astype(np.float32)
        group_r = pick_first_available(
            f,
            ['Group/Group_R_Crit200', 'Group_R_TopHat200', 'Group/Group_R_Mean200']
        ).astype(np.float32)

    # Se GroupMass è in unità 1e10 Msun/h, converte in Msun/h.
    # Nei cataloghi CAMELS/TNG è comune avere masse in 1e10 Msun/h.
    if np.nanmedian(group_mass) < 1e6:
        group_mass_msun_h = group_mass * 1e10
    else:
        group_mass_msun_h = group_mass

    mask = group_mass_msun_h >= mass_cut
    halos = {
        'pos': group_pos[mask],
        'mass_msun_h': group_mass_msun_h[mask],
        'r': group_r[mask],
    }
    return halos

In [ ]:
dm = load_dm_snapshot(SNAPSHOT_PATH)
halos = load_halos(GROUPS_PATH, MASS_CUT)

coords = dm['coords']
vels = dm['vels']
masses = dm['masses']
m_particle = dm['m_particle']
box_size = dm['box_size']

print(f'Particelle DM: {coords.shape[0]:,}')
print(f'Massa particella: {m_particle:.4e} Msun/h')
print(f'BoxSize: {box_size:.2f} ckpc/h')
print(f'Aloni sopra MASS_CUT: {halos["pos"].shape[0]:,}')

## 4) Smoothing lengths SPH (con cache)

Calcolo costoso: stimiamo `h` una sola volta con `k=32` e salviamo su disco in `.npy`.

In [ ]:
h_cache_path = CACHE_ROOT / f'h_sph_k{K_NEIGHBORS}.npy'

if h_cache_path.exists():
    h_sph = np.load(h_cache_path)
    print(f'Caricato smoothing lengths da cache: {h_cache_path}')
else:
    h_sph = sphviewer2.estimate_h(
        coords[:, 0], coords[:, 1], coords[:, 2],
        Lbox=box_size,
        k=K_NEIGHBORS
    ).astype(np.float32)
    np.save(h_cache_path, h_sph)
    print(f'Salvato smoothing lengths: {h_cache_path}')

print('h_sph stats:', np.min(h_sph), np.median(h_sph), np.max(h_sph))

## 5) Utility per slicing, rendering e maschere target

Per ogni fetta lungo Z:
1. selezioniamo le particelle nel range
2. renderizziamo la mappa SPH densità 2D `1024x1024`
3. creiamo una maschera target con i pixel interni ai raggi degli aloni che intersecano la fetta.

In [ ]:
def z_slice_edges(box_size, n_slices):
    return np.linspace(0.0, box_size, n_slices + 1, dtype=np.float32)


def render_sph_slice(coords_xyz, masses, hvals, box_size, resolution=1024, r_max=10, target_cells_per_h=4):
    # API coerente con descrizione del prof; adatta eventuali keyword se la tua versione differisce.
    img = sphviewer2.render(
        coords_xyz[:, 0],
        coords_xyz[:, 1],
        coords_xyz[:, 2],
        masses,
        hvals,
        Lbox=box_size,
        r_max=r_max,
        target_cells_per_h=target_cells_per_h,
    )

    img = np.asarray(img, dtype=np.float32)
    if img.shape != (resolution, resolution):
        raise ValueError(f'Render SPH shape inattesa: {img.shape}, atteso {(resolution, resolution)}')
    return img


def halo_mask_for_slice(halo_pos, halo_r, z_min, z_max, box_size, resolution=1024):
    """
    Proiezione XY: per ogni alone che interseca la fetta Z,
    disegna il cerchio con r_proj = sqrt(r^2 - dz^2).
    """
    mask = np.zeros((resolution, resolution), dtype=np.uint8)

    z_center = 0.5 * (z_min + z_max)
    half_thickness = 0.5 * (z_max - z_min)

    # distanza periodica lungo z
    dz = np.abs(halo_pos[:, 2] - z_center)
    dz = np.minimum(dz, box_size - dz)

    intersects = dz <= (halo_r + half_thickness)
    pos_i = halo_pos[intersects]
    r_i = halo_r[intersects]
    dz_i = dz[intersects]

    if pos_i.shape[0] == 0:
        return mask

    x = np.arange(resolution, dtype=np.float32)
    y = np.arange(resolution, dtype=np.float32)
    xx, yy = np.meshgrid(x, y, indexing='xy')

    # coordinate pixel -> ckpc/h
    pix_to_phys = box_size / resolution

    for p, r3d, dzv in zip(pos_i, r_i, dz_i):
        radial_component_sq = float(r3d**2 - dzv**2)
        if radial_component_sq <= 0:
            continue
        r_proj = math.sqrt(radial_component_sq)

        cx = p[0] / pix_to_phys
        cy = p[1] / pix_to_phys

        # distanza periodica in pixel
        dx = np.abs(xx - cx)
        dy = np.abs(yy - cy)
        dx = np.minimum(dx, resolution - dx)
        dy = np.minimum(dy, resolution - dy)

        inside = (dx * pix_to_phys) ** 2 + (dy * pix_to_phys) ** 2 <= r_proj**2
        mask[inside] = 1

    return mask

## 6) Generazione mappe complete (1024x1024)

Salviamo per ogni fetta Z:
- `density_slice_XXX.npy`
- `mask_slice_XXX.npy`

In parallelo salviamo un file metadata JSON con parametri usati.

In [ ]:
slice_edges = z_slice_edges(box_size, N_SLICES_Z)

for i in range(N_SLICES_Z):
    z_min, z_max = slice_edges[i], slice_edges[i + 1]
    sel = (coords[:, 2] >= z_min) & (coords[:, 2] < z_max)

    coords_i = coords[sel]
    masses_i = masses[sel]
    h_i = h_sph[sel]

    if coords_i.shape[0] == 0:
        density = np.zeros((SPH_RESOLUTION, SPH_RESOLUTION), dtype=np.float32)
    else:
        density = render_sph_slice(
            coords_i, masses_i, h_i, box_size,
            resolution=SPH_RESOLUTION,
            r_max=SPH_R_MAX,
            target_cells_per_h=TARGET_CELLS_PER_H
        )

    mask = halo_mask_for_slice(
        halos['pos'], halos['r'],
        z_min, z_max,
        box_size,
        resolution=SPH_RESOLUTION
    )

    np.save(MAPS_ROOT / f'density_slice_{i:03d}.npy', density)
    np.save(MAPS_ROOT / f'mask_slice_{i:03d}.npy', mask)

metadata = {
    **config,
    'box_size_ckpc_h': float(box_size),
    'm_particle_msun_h': float(m_particle),
    'n_particles': int(coords.shape[0]),
    'n_halos_after_cut': int(halos['pos'].shape[0]),
}

with open(MAPS_ROOT / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print(f'Mappe salvate in: {MAPS_ROOT}')

## 7) Estrazione patch 256x256 per training

Da ogni coppia (`density`, `mask`) estraiamo patch con stride fisso.

Output finale:
- `X_train.npy` (N, 256, 256, 1)
- `y_train.npy` (N, 256, 256, 1)
- `patch_index.json` per tracciabilità (slice e coordinate origine patch)

In [ ]:
def extract_patches_2d(arr, patch_size=256, stride=256):
    h, w = arr.shape
    patches = []
    coords = []
    for y0 in range(0, h - patch_size + 1, stride):
        for x0 in range(0, w - patch_size + 1, stride):
            p = arr[y0:y0 + patch_size, x0:x0 + patch_size]
            patches.append(p)
            coords.append((x0, y0))
    return patches, coords


X_list = []
y_list = []
index_rows = []

for i in range(N_SLICES_Z):
    density = np.load(MAPS_ROOT / f'density_slice_{i:03d}.npy')
    mask = np.load(MAPS_ROOT / f'mask_slice_{i:03d}.npy')

    d_patches, patch_coords = extract_patches_2d(density, PATCH_SIZE, PATCH_STRIDE)
    m_patches, _ = extract_patches_2d(mask, PATCH_SIZE, PATCH_STRIDE)

    for j, (dp, mp, (x0, y0)) in enumerate(zip(d_patches, m_patches, patch_coords)):
        # Normalizzazione robusta su ogni patch densità
        p1, p99 = np.percentile(dp, [1, 99])
        denom = max(p99 - p1, 1e-8)
        dp_norm = np.clip((dp - p1) / denom, 0, 1).astype(np.float32)

        X_list.append(dp_norm[..., None])
        y_list.append((mp > 0).astype(np.uint8)[..., None])

        index_rows.append({
            'slice_id': i,
            'patch_id_in_slice': j,
            'x0': int(x0),
            'y0': int(y0),
        })

X_train = np.stack(X_list, axis=0)
y_train = np.stack(y_list, axis=0)

np.save(PATCH_ROOT / 'X_train.npy', X_train)
np.save(PATCH_ROOT / 'y_train.npy', y_train)

with open(PATCH_ROOT / 'patch_index.json', 'w', encoding='utf-8') as f:
    json.dump(index_rows, f, indent=2)

print('Shape X_train:', X_train.shape)
print('Shape y_train:', y_train.shape)
print(f'Patch dataset salvato in: {PATCH_ROOT}')

## 8) Quality check veloce

Visualizzazione rapida di alcune patch per controllare consistenza input/target.

In [ ]:
n_show = 4
ids = np.linspace(0, len(X_train) - 1, n_show, dtype=int)

fig, axes = plt.subplots(n_show, 2, figsize=(8, 3 * n_show))
for r, idx in enumerate(ids):
    axes[r, 0].imshow(X_train[idx, ..., 0], origin='lower', cmap='magma')
    axes[r, 0].set_title(f'Input density patch #{idx}')
    axes[r, 0].axis('off')

    axes[r, 1].imshow(y_train[idx, ..., 0], origin='lower', cmap='gray')
    axes[r, 1].set_title(f'Target mask patch #{idx}')
    axes[r, 1].axis('off')

plt.tight_layout()
plt.show()

## 9) Step successivo

Con i file `X_train.npy` e `y_train.npy` puoi passare al notebook di training U-Net.

Estensioni naturali (in linea col piano tesi):
- aggiungere canali velocità (es. modulo velocità o componenti proiettate)
- passare da segmentazione pura a task multi-output (mask + regressione centro/raggio)
- validare il catalogo predetto con Halo Mass Function.